# Clusterização de Hospitais por Perfil Operacional

Em vez de tratar cada hospital individualmente, agrupo unidades com padrão parecido de ocupação, permanência, mortalidade e readmissão — essa é a parte de "otimização de recursos" do escopo do projeto. Com os hospitais agrupados, a rede pode aplicar a mesma estratégia de gestão (redistribuição de leitos, reforço de equipe, programa de acompanhamento pós-alta) a cada grupo, em vez de precisar de um plano individual para cada uma das unidades.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.clusterizacao_hospitais import (
    preparar_dados_cluster, escolher_k, treinar_kmeans, descrever_clusters
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

indicadores = pd.read_parquet('../data/processed/indicadores_hospital.parquet')
indicadores[['hospital_id', 'tipo_gestao', 'tempo_medio_permanencia', 'taxa_readmissao', 'taxa_ocupacao_media', 'rotatividade_leitos']]

## 1. Padronizando os indicadores

Sem padronizar, ocupação (uma proporção entre 0 e ~1,2) e valor médio de internação (milhares de reais) teriam pesos completamente diferentes na distância euclidiana usada pelo K-Means.

In [ ]:
X_escalado, scaler = preparar_dados_cluster(indicadores)
X_escalado.shape

## 2. Escolhendo K

Com só 16 hospitais na rede simulada, não espero um cotovelo muito nítido na curva de inércia — por isso confirmo a escolha com o silhouette score, que mede o quanto cada hospital está mais próximo do seu próprio cluster do que dos outros.

In [ ]:
tabela_k = escolher_k(X_escalado)
tabela_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tabela_k['k'], tabela_k['inercia'], marker='o')
axes[0].set_title('Método do cotovelo')
axes[0].set_xlabel('K')
axes[1].plot(tabela_k['k'], tabela_k['silhouette'], marker='o', color='crimson')
axes[1].set_title('Silhouette score')
axes[1].set_xlabel('K')
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_escolha_k.png', dpi=150)
plt.show()

melhor_k = int(tabela_k.loc[tabela_k['silhouette'].idxmax(), 'k'])
print(f'K escolhido: {melhor_k}')

## 3. Treinando o K-Means final

In [ ]:
modelo_km = treinar_kmeans(X_escalado, melhor_k)
perfil = descrever_clusters(indicadores, modelo_km.labels_)
perfil

## 4. Visualizando os clusters

Ocupação e readmissão são os dois eixos mais ligados ao problema de negócio do projeto (superlotação e qualidade do cuidado pós-alta), então uso os dois no eixo X e Y. O tamanho de cada bolha é o volume de internações do hospital.

In [ ]:
fig, ax = plt.subplots()
scatter = ax.scatter(
    indicadores['taxa_ocupacao_media'], indicadores['taxa_readmissao'],
    c=modelo_km.labels_, cmap='viridis', s=indicadores['n_internacoes'] / 10, alpha=0.8
)
ax.set_xlabel('Taxa de ocupação média')
ax.set_ylabel('Taxa de readmissão')
ax.set_title('Clusters de hospitais por perfil operacional')
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_clusters.png', dpi=150)
plt.show()

## 5. Dando nome de negócio a cada cluster

Olhando o perfil médio de cada grupo, consigo nomear o que cada um representa na prática:

- **Crise aguda de leitos** — ocupação acima de 115% e a maior taxa de readmissão do grupo. É o cenário mais crítico: hospital sobrecarregado *e* dando alta antes do ideal, o que provavelmente alimenta a própria readmissão.
- **Superlotação crônica** — ocupação em torno de 100%, no limite da capacidade, mas com readmissão dentro da média da rede. É o maior grupo em número de hospitais — sinaliza que operar no limite já é a norma, não a exceção, nessa rede simulada.
- **Alta demanda sob controle** — hospitais com o maior volume médio de internações, mas ocupação abaixo de 85%. Mostra que porte grande não significa necessariamente sobrecarga, quando a capacidade foi dimensionada de forma adequada.
- **Operação equilibrada** — ocupação moderada (por volta de 75%) e indicadores dentro da média em todos os eixos. É o grupo de referência para comparar os demais.
- **Baixa demanda, mortalidade mais alta** — o hospital com menor ocupação da rede, mas com a maior taxa de mortalidade entre os clusters. Vale investigar se é um efeito de amostra pequena (poucos casos) ou um padrão real de case-mix mais grave para o volume que atende.

Essa segmentação é o que eu usaria para decidir onde a rede deveria agir primeiro: os hospitais do grupo de crise aguda são candidatos naturais a reforço de leitos ou redistribuição de pacientes vindos de outras unidades da mesma região.

## Conclusões

A clusterização confirma o que já aparecia na EDA: a rede não sofre de superlotação de forma uniforme — há uma minoria de hospitais em situação claramente mais crítica, e é nela que uma intervenção prioritária (reforço de leitos, revisão de altas, programa de acompanhamento pós-alta) tem o maior potencial de impacto por unidade de esforço.

Isso encerra a parte de modelagem do projeto. O relatório final, em `reports/relatorio_final.md`, consolida os achados dos quatro notebooks em uma visão única, com foco nas recomendações de negócio.